# Analyzing 10-K Filings with an LLM API

In the previous notebook (`read-edgar-filings.ipynb`) we used `edgartools` to download and clean three sections of Apple's most recent 10-K — _Business_, _Risk Factors_, and _Management's Discussion and Analysis_ — and saved them to `extracted/`.

In this notebook we will feed those sections into a Large Language Model through an **OpenAI-compatible API** and run several analytical tasks:

| #   | Task                                                         | Uses JSON schema? |
| --- | ------------------------------------------------------------ | ----------------- |
| 1   | One-paragraph executive summary                              | ❌ free-form      |
| 2   | Plain-English explanation of MD&A                            | ❌ free-form      |
| 3   | Sentiment narrative on MD&A                                  | ❌ free-form      |
| 4   | Risk-factor classification                                   | ✅ JSON schema    |
| 5   | Named-entity extraction (products, competitors, geographies) | ✅ JSON schema    |
| 6   | Final structured "executive briefing"                        | ✅ JSON schema    |

Tasks 4–6 use a **structured response schema** so the model returns clean JSON we can parse and use downstream — no string scraping required.

We deliberately use a small model (`gpt-5.4-mini` by default, falling back to `gpt-5.4-nano` for the cheapest tasks). You can swap in any OpenAI-compatible model — see the **Switching providers** section below.


## 1. Install and import

You only need to run the install once.


In [20]:
# %pip install --quiet "openai>=1.40.0"

In [21]:
import json
import os
from pathlib import Path
from textwrap import shorten

from openai import OpenAI

## 2. Configure the OpenAI-compatible client

The `openai` Python SDK accepts two arguments that determine _which provider_ you are talking to:

- `api_key` — your secret key for that provider.
- `base_url` — the HTTPS endpoint the SDK should send requests to.

Leave `base_url` unset to talk to OpenAI directly. Set it to `https://openrouter.ai/api/v1` to talk to **OpenRouter**, or to any other OpenAI-compatible provider's URL.

You can create a local `openai_config.json` file in the same folder with this notebook with the following keys, and the notebook will read from that if it exists.

`openai_config.json` for OpenAI

```json
{
  "OPENAI_API_KEY": "your-openai-api-key-here",
  "OPENAI_BASE_URL": "https://api.openai.com/v1"
}
```

`openai_config.json` for OpenRouter

```json
{
  "OPENAI_API_KEY": "your-openrouter-api-key-here",
  "OPENAI_BASE_URL": "https://openrouter.ai/api/v1"
}
```

Alternatively, you can read both values from environment variables.

```bash
# Option A — OpenAI directly
export OPENAI_API_KEY="sk-..."

# Option B — OpenRouter (or any other compatible provider)
export OPENAI_API_KEY="sk-or-v1-..."
export OPENAI_BASE_URL="https://openrouter.ai/api/v1"
```

:::warning Never share your API keys publicly

Never share your API keys publicly, and avoid hardcoding them directly in code. Use environment variables or local configuration files that are excluded from version control (e.g., via `.gitignore`) to keep your keys secure.

You shouldn't even provide these keys to your classmates or the instructional team since they can be used to make requests that you will be billed for. If you want to share your work, clearly indicate that API calls will not work without valid keys.

:::


In [22]:
api_key = os.environ.get("OPENAI_API_KEY")
base_url = os.environ.get("OPENAI_BASE_URL")

# Fallback to config file if needed
if not api_key or not base_url:
    try:
        with open("openai_config.json", "r") as f:
            config = json.load(f)

        if not api_key:
            api_key = config.get("OPENAI_API_KEY")

        if not base_url:
            base_url = config.get("OPENAI_BASE_URL")

    except FileNotFoundError:
        pass  # Silently ignore if config file doesn't exist

# Validate API key
if not api_key:
    raise RuntimeError(
        "Please set OPENAI_API_KEY as an environment variable or in openai_config.json."
    )

client = OpenAI(api_key=api_key, base_url=base_url)

print("Talking to:", base_url or "https://api.openai.com/v1 (OpenAI default)")

Talking to: https://api.openai.com/v1


### Choose a model

For routine financial-text tasks like summarization and extraction, a small, cheap model is plenty. We default to `gpt-5.4-mini` and use `gpt-5.4-nano` for the very cheapest task. If you are on OpenRouter, replace these with a model identifier that exists on OpenRouter, e.g. `openai/gpt-5.4-mini`, `meta-llama/llama-3.1-8b-instruct`, `google/gemini-2.5-flash`, etc.


In [23]:
DEFAULT_MODEL = os.environ.get("LLM_MODEL", "gpt-5.4-mini")
CHEAP_MODEL = os.environ.get("LLM_MODEL_CHEAP", "gpt-5.4-nano")

print("Default model:", DEFAULT_MODEL)
print("Cheap model  :", CHEAP_MODEL)

Default model: gpt-5.4-mini
Cheap model  : gpt-5.4-nano


## 3. Load the extracted 10-K sections

We expect the previous notebook to have written three files into `extracted/`. If you skipped that notebook, re-run it first.


In [24]:
TICKER = "AAPL"
extracted_dir = Path("extracted")


def load_section(name: str) -> str:
    path = extracted_dir / f"{TICKER}_{name}.txt"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run `read-edgar-filings.ipynb` first to populate `extracted/`."
        )
    return path.read_text(encoding="utf-8")


business = load_section("business")
risks = load_section("risk_factors")
mdna = load_section("mdna")

for name, text in [("Business", business), ("Risk Factors", risks), ("MD&A", mdna)]:
    print(
        f"{name:<13} {len(text):>7,} chars  -- {shorten(text.replace(chr(10), ' '), width=120)}"
    )

Business       16,216 chars  -- # Source: SEC EDGAR 0000320193-25-000079 (10-K) # Company: Apple Inc. (CIK 320193, ticker AAPL) # Filed: [...]
Risk Factors   68,329 chars  -- # Source: SEC EDGAR 0000320193-25-000079 (10-K) # Company: Apple Inc. (CIK 320193, ticker AAPL) # Filed: [...]
MD&A           18,176 chars  -- # Source: SEC EDGAR 0000320193-25-000079 (10-K) # Company: Apple Inc. (CIK 320193, ticker AAPL) # Filed: [...]


### Helper: trim long sections

LLM context windows are large but not free. We truncate any single section to ~12,000 characters (~3,000 tokens) so each call is fast and cheap. For real production work you would chunk + summarize iteratively instead of truncating; we keep things simple here for teaching.


In [25]:
MAX_CHARS = 12_000


def trim(text: str, max_chars: int = MAX_CHARS) -> str:
    if len(text) <= max_chars:
        return text
    head = text[: max_chars // 2]
    tail = text[-max_chars // 2 :]
    return head + "\n\n[... truncated for brevity ...]\n\n" + tail


business_t = trim(business)
risks_t = trim(risks)
mdna_t = trim(mdna)

## 4. A small wrapper around `chat.completions.create`

Everything in this notebook is a chat-completion call. To avoid repetition we wrap the call in a helper that:

1. Always passes `model`, `messages`, and (optionally) a `response_format` for structured output.
2. Returns the assistant message _and_ the token-usage report so we can keep an eye on cost.


In [26]:
def chat(
    user_prompt: str,
    *,
    system: str = "You are a careful financial analyst who answers concisely.",
    model: str = DEFAULT_MODEL,
    response_format: dict | None = None,
    temperature: float = 0.2,
) -> tuple[str, dict]:
    """Send one chat-completion request and return (text, usage_dict)."""
    kwargs: dict = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
    }
    if response_format is not None:
        kwargs["response_format"] = response_format

    resp = client.chat.completions.create(**kwargs)
    text = resp.choices[0].message.content or ""
    usage = {
        "input_tokens": resp.usage.prompt_tokens,
        "output_tokens": resp.usage.completion_tokens,
        "total_tokens": resp.usage.total_tokens,
    }
    return text, usage

## 5. Free-form task #1 — One-paragraph executive summary

We start with the simplest possible task: ask the model for a short, narrative summary of the **Business** section. No schema, no constraints — just plain prose.


In [27]:
prompt = (
    "Write a single, tight paragraph (no more than 120 words) that explains, in plain "
    "language, what business this company is in. Avoid jargon and avoid quoting the "
    "filing verbatim.\n\n"
    "=== BUSINESS SECTION ===\n"
    f"{business_t}"
)

summary, usage = chat(prompt)
print(summary)
print("\nUsage:", usage)

Apple makes and sells consumer technology products and the software and services that go with them. Its main products are the iPhone, Mac computers, iPad tablets, Apple Watch, AirPods, and other devices like Apple TV and Vision Pro. It also earns money from services such as the App Store, cloud storage, music, video, fitness, news, advertising, AppleCare support, and payment tools like Apple Pay and Apple Card. Apple sells directly through its stores, website, and sales teams, as well as through carriers and other resellers, serving consumers, schools, businesses, and governments around the world.

Usage: {'input_tokens': 2310, 'output_tokens': 126, 'total_tokens': 2436}


## 6. Free-form task #2 — Plain-English MD&A explainer

The _Management's Discussion and Analysis_ section is dense and full of accounting language. Ask the model to explain it as if to a smart business student who has never read a 10-K.


In [28]:
prompt = (
    "You are tutoring a college student who has never read a 10-K before. Read the "
    "following Management's Discussion and Analysis (MD&A) section and explain, in "
    "5–7 short bullet points, what management is saying about how the business "
    "performed and why. Use plain English; define any accounting terms you use.\n\n"
    "=== MD&A ===\n"
    f"{mdna_t}"
)

mdna_explainer, usage = chat(prompt)
print(mdna_explainer)
print("\nUsage:", usage)

- **Apple grew overall in 2025.** Total net sales were **$416.2 billion**, up **6%** from 2024. In plain English, Apple sold more overall than it did the year before.

- **The biggest drivers were iPhone and Services.** Apple says higher sales of **iPhone** and **Services** helped results. **Services** means things like the App Store, subscriptions, iCloud, AppleCare, and other recurring digital offerings.

- **Services was a standout.** Services revenue rose **14%**, which suggests Apple is making more money from its ecosystem, not just from hardware sales.

- **Mac also did well; Wearables declined.** **Mac** sales rose **12%**, while **Wearables, Home and Accessories** fell **4%**. So some product lines were strong, but not all.

- **Results varied by region.** Sales grew in the **Americas, Europe, Japan, and Rest of Asia Pacific**, but **Greater China** sales fell **4%**. Apple says China’s decline was mainly because of lower iPhone sales.

- **Currency and trade issues are a risk.

## 7. Free-form task #3 — Sentiment narrative on MD&A

A classic financial-NLP task: how _positive_ or _negative_ is management's tone? We ask for a short narrative judgment plus a 1–5 score embedded in the prose. We will redo this task with strict structured output in section 9 so you can compare the two styles.


In [29]:
prompt = (
    "Read the MD&A excerpt below. In 3–4 sentences, describe management's overall "
    "tone (optimistic, neutral, cautious, defensive, etc.) and end your answer with "
    "a single line of the form `Score: X/5` where 1 = very negative and 5 = very "
    "positive.\n\n"
    "=== MD&A ===\n"
    f"{mdna_t}"
)

sentiment_text, usage = chat(prompt, model=CHEAP_MODEL)
print(sentiment_text)
print("\nUsage:", usage)

Management’s tone is largely **cautiously neutral**: it highlights steady growth in net sales and strength in categories like iPhone and Services, but repeatedly emphasizes uncertainty and downside risks. The MD&A stresses potential material adverse impacts from **macroeconomic conditions** (inflation, FX, interest rates) and especially **tariffs/trade disputes**, noting the ultimate effects are uncertain. It also maintains a measured stance on **tax and legal contingencies**, stating outcomes could differ from reserves and expectations. Overall, the tone balances performance updates with risk-focused caution.  
Score: 3/5

Usage: {'input_tokens': 2826, 'output_tokens': 122, 'total_tokens': 2948}


## 8. Structured task #1 — Risk-factor classification (JSON schema)

Now we shift to **structured output**. We give the model a JSON schema that forces the response to be a list of objects, each with a `theme`, a `severity` (low/medium/high), and a one-sentence `description`. Because the schema is enforced, we can `json.loads()` the response and use it directly.

We use the OpenAI [JSON Schema response format](https://platform.openai.com/docs/guides/structured-outputs):

```python
response_format = {
    "type": "json_schema",
    "json_schema": {"name": "...", "strict": True, "schema": {...}},
}
```

Most OpenAI-compatible providers (including OpenRouter routes that proxy to OpenAI/Mistral/Gemini-class models) honor the same shape. If your chosen model does not support structured outputs, you can fall back to `response_format={"type": "json_object"}` plus an explicit instruction in the prompt to return JSON.


In [30]:
risk_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "risk_classification",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "required": ["risks"],
            "properties": {
                "risks": {
                    "type": "array",
                    "minItems": 5,
                    "maxItems": 8,
                    "items": {
                        "type": "object",
                        "additionalProperties": False,
                        "required": ["theme", "severity", "description"],
                        "properties": {
                            "theme": {"type": "string"},
                            "severity": {
                                "type": "string",
                                "enum": ["low", "medium", "high"],
                            },
                            "description": {"type": "string"},
                        },
                    },
                }
            },
        },
    },
}

prompt = (
    "Read the Risk Factors section below and identify 5–8 distinct risk THEMES "
    "(e.g., 'Supply chain concentration', 'Foreign exchange exposure', "
    "'Regulatory / antitrust'). For each theme, assign a severity (low/medium/high) "
    "based on how much space and emphasis the filing gives it, and write a "
    "one-sentence plain-English description.\n\n"
    "=== RISK FACTORS ===\n"
    f"{risks_t}"
)

risk_json_text, usage = chat(prompt, response_format=risk_schema)
print(risk_json_text[:500], "...")
print("\nUsage:", usage)

{"risks":[{"theme":"Global macroeconomic slowdown and consumer demand","severity":"high","description":"Weak growth, recession, inflation, higher rates, and lower consumer confidence could reduce demand for Apple’s products and services and hurt results."},{"theme":"Geopolitical, trade, and supply-chain disruption","severity":"high","description":"Tariffs, export/import restrictions, conflicts, and other international disruptions could raise costs, limit product availability, and force expensive ...

Usage: {'input_tokens': 2304, 'output_tokens': 349, 'total_tokens': 2653}


In [31]:
risk_data = json.loads(risk_json_text)
risk_data["risks"][:3]

[{'theme': 'Global macroeconomic slowdown and consumer demand',
  'severity': 'high',
  'description': 'Weak growth, recession, inflation, higher rates, and lower consumer confidence could reduce demand for Apple’s products and services and hurt results.'},
 {'theme': 'Geopolitical, trade, and supply-chain disruption',
  'severity': 'high',
  'description': 'Tariffs, export/import restrictions, conflicts, and other international disruptions could raise costs, limit product availability, and force expensive supply-chain changes.'},
 {'theme': 'International manufacturing and supplier concentration',
  'severity': 'high',
  'description': 'Because much of Apple’s manufacturing and key suppliers are concentrated outside the U.S., disruptions in countries like China, India, Taiwan, and Vietnam could materially affect operations.'}]

Because the response is structured, we can drop it straight into a DataFrame and sort, filter, or chart it like any other tabular data.


In [32]:
import pandas as pd

risk_df = pd.DataFrame(risk_data["risks"])
risk_df = risk_df.sort_values(
    "severity",
    key=lambda s: s.map({"high": 0, "medium": 1, "low": 2}),
).reset_index(drop=True)
risk_df

,theme,severity,description
0,Global macroeconomic slowdown and consumer demand,high,"Weak growth, recession, inflation, higher rate..."
1,"Geopolitical, trade, and supply-chain disruption",high,"Tariffs, export/import restrictions, conflicts..."
2,International manufacturing and supplier conce...,high,Because much of Apple’s manufacturing and key ...
3,Foreign exchange exposure,medium,Movements in currency exchange rates can reduc...
4,Credit and counterparty risk,medium,"Apple could face losses if customers, carriers..."
5,Investment portfolio and liquidity risk,medium,"Changes in interest rates, credit quality, mar..."
6,Tax law and tax authority uncertainty,medium,"Changes in tax rates, new global tax rules, or..."
7,Stock price volatility and market expectations,medium,Apple’s share price may fall sharply if invest...


## 9. Structured task #2 — Named-entity extraction

We ask the model to pull three lists of entities out of the _Business_ section:

- **products** mentioned by name,
- **competitors** explicitly named, and
- **geographies** (countries / regions) the company operates in.

Each list element includes a short `evidence` snippet so the user can audit where the model saw the entity in the source text.


In [33]:
entity_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "entity_extraction",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "required": ["products", "competitors", "geographies"],
            "properties": {
                bucket: {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "additionalProperties": False,
                        "required": ["name", "evidence"],
                        "properties": {
                            "name": {"type": "string"},
                            "evidence": {"type": "string"},
                        },
                    },
                }
                for bucket in ("products", "competitors", "geographies")
            },
        },
    },
}

prompt = (
    "From the Business section below, extract three lists of entities:\n"
    "- products       : specific product or service names the company sells\n"
    "- competitors    : companies explicitly identified as competitors\n"
    "- geographies    : countries, regions, or markets the company explicitly operates in\n\n"
    "For each entity, include a short `evidence` snippet (<= 20 words) quoting the "
    "phrase from the filing that supports the extraction. Only include entities that "
    "are explicitly mentioned in the text — do NOT guess.\n\n"
    "=== BUSINESS SECTION ===\n"
    f"{business_t}"
)

entity_json_text, usage = chat(prompt, response_format=entity_schema)
entities = json.loads(entity_json_text)

print("Products    :", len(entities["products"]))
print("Competitors :", len(entities["competitors"]))
print("Geographies :", len(entities["geographies"]))
print("\nUsage:", usage)

Products    : 17
Competitors : 0
Geographies : 15

Usage: {'input_tokens': 2479, 'output_tokens': 657, 'total_tokens': 3136}


In [34]:
# Quick peek
for bucket in ("products", "competitors", "geographies"):
    print(f"\n--- {bucket.upper()} ---")
    for e in entities[bucket][:5]:
        print(f"  {e['name']!r:35}  evidence: {e['evidence']!r}")


--- PRODUCTS ---
  'iPhone'                             evidence: '“iPhone® is the Company’s line of smartphones”'
  'Mac'                                evidence: '“Mac® is the Company’s line of personal computers”'
  'iPad'                               evidence: '“iPad® is the Company’s line of multipurpose tablets”'
  'Apple Watch'                        evidence: '“includes Apple Watch® Series 11”'
  'AirPods'                            evidence: '“The Company’s line of wireless headphones includes AirPods®”'

--- COMPETITORS ---

--- GEOGRAPHIES ---
  'Americas'                           evidence: '“reportable segments consist of the Americas”'
  'Europe'                             evidence: '“reportable segments consist of ... Europe”'
  'Greater China'                      evidence: '“reportable segments consist of ... Greater China”'
  'Japan'                              evidence: '“reportable segments consist of ... Japan”'
  'Rest of Asia Pacific'               evidence: 

## 10. Structured task #3 — Final "executive briefing"

For the capstone task we combine everything into a single structured _executive briefing_: a short summary of the company, a tone score for management, the top 3 risks, and a confidence level. This is the kind of object an analytics team would dump into a database, a Slack message, or a dashboard.


In [35]:
briefing_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "executive_briefing",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "required": [
                "company_one_liner",
                "management_tone",
                "tone_score",
                "top_risks",
                "confidence",
            ],
            "properties": {
                "company_one_liner": {"type": "string"},
                "management_tone": {
                    "type": "string",
                    "enum": [
                        "very negative",
                        "negative",
                        "neutral",
                        "positive",
                        "very positive",
                    ],
                },
                "tone_score": {"type": "integer", "minimum": 1, "maximum": 5},
                "top_risks": {
                    "type": "array",
                    "minItems": 3,
                    "maxItems": 3,
                    "items": {"type": "string"},
                },
                "confidence": {
                    "type": "string",
                    "enum": ["low", "medium", "high"],
                },
            },
        },
    },
}

# We give the model just the previously extracted artifacts — no need to re-send the full filing.
prompt = (
    "Produce a one-paragraph executive briefing for a portfolio manager. Base your "
    "answer ONLY on the materials below.\n\n"
    "=== BUSINESS SUMMARY (from a previous LLM call) ===\n"
    f"{summary}\n\n"
    "=== MD&A EXPLAINER (from a previous LLM call) ===\n"
    f"{mdna_explainer}\n\n"
    "=== RISK CLASSIFICATION (JSON, from a previous LLM call) ===\n"
    f"{json.dumps(risk_data, indent=2)}"
)

briefing_text, usage = chat(prompt, response_format=briefing_schema)
briefing = json.loads(briefing_text)
briefing

{'company_one_liner': 'Apple is a global consumer technology and services company with a large installed base, led by iPhone and an increasingly important recurring Services ecosystem.',
 'management_tone': 'positive',
 'tone_score': 3,
 'top_risks': ['Global macroeconomic slowdown and weaker consumer demand could pressure device and services sales.',
  'Geopolitical, trade, and supply-chain disruptions, including tariffs, could raise costs and constrain product availability.',
  'Foreign exchange movements and regional weakness, especially in Greater China, could weigh on reported results and margins.'],
 'confidence': 'high'}

Because every field has a known type, downstream code can act on it without parsing:

```python
if briefing["tone_score"] <= 2 and briefing["confidence"] != "low":
    alert_compliance_team(briefing)
```


## 11. Cost awareness

Token usage is reported on every response under `resp.usage`. Below we estimate the cost of running this entire notebook end-to-end at the time-of-writing list price for `gpt-5.4-mini`. **Always update the prices to whatever your provider currently charges** — model prices change frequently.


In [36]:
# Example list prices in USD per 1M tokens. UPDATE BEFORE RELYING ON THIS.
PRICE_PER_1M = {
    "gpt-5.4-mini": {"input": 0.15, "output": 0.60},
    "gpt-5.4-nano": {"input": 0.05, "output": 0.20},
}


def estimate_cost(usage: dict, model: str) -> float:
    p = PRICE_PER_1M.get(model, {"input": 0.15, "output": 0.60})
    return (usage["input_tokens"] / 1_000_000) * p["input"] + (
        usage["output_tokens"] / 1_000_000
    ) * p["output"]


# Demonstrate on the most recent call.
print(f"Last call usage: {usage}")
print(f"Estimated cost : ${estimate_cost(usage, DEFAULT_MODEL):.6f}")

Last call usage: {'input_tokens': 1107, 'output_tokens': 121, 'total_tokens': 1228}
Estimated cost : $0.000239


:::{tip} Watch the ratio
Output tokens are typically 3–5× more expensive than input tokens. If you find yourself paying mostly for _output_ (e.g., asking the model to repeat the filing back to you), redesign the prompt to ask for a _summary_ or a _structured extraction_ instead.
:::


## 12. Switching providers (OpenRouter & friends)

Because the `openai` SDK is just an HTTP client with a particular request shape, you can point it at any OpenAI-compatible provider by changing two values:

```python
client = OpenAI(
    api_key  = os.environ["OPENROUTER_API_KEY"],
    base_url = "https://openrouter.ai/api/v1",
)

text, usage = client.chat.completions.create(
    model    = "openai/gpt-5.4-mini",   # OpenRouter's slug for the same model
    messages = [...],
).choices[0].message.content, ...
```

Every helper above (`chat`, `risk_schema`, `entity_schema`, `briefing_schema`) continues to work unchanged. If your target model on OpenRouter does not support strict JSON schemas, downgrade `response_format` to `{"type": "json_object"}` and add an explicit _"Respond ONLY with valid JSON matching this shape: {…}"_ instruction in the user prompt.

:::{seealso} Browse OpenRouter models
The full list of available models, with live per-token prices, lives at <https://openrouter.ai/models>. Filter by _Tools_ and _JSON mode_ support if you plan to rely on structured outputs.
:::


## 13. Recap

In this notebook we:

- Connected the `openai` Python client to either OpenAI or any OpenAI-compatible provider by setting `api_key` and `base_url`.
- Ran **three free-form** chat-completion tasks: business summary, MD&A explainer, and a sentiment narrative.
- Ran **three structured** tasks using JSON Schema response formats: risk classification, entity extraction, and a final executive briefing.
- Tracked token usage on every call and estimated cost from a per-million-token price table.

Combined with the EDGAR notebook, you now have a tiny end-to-end pipeline that turns a raw 10-K into machine-readable analytical artifacts — the same pattern used by financial research firms, audit teams, and investor-relations groups to triage thousands of filings per quarter.
